# The cost of a cut

This notebook runs from the competition CSVs. Set `BDB_EXECUTE=1` to repeat extraction, development, the original freeze recipe, and the 2025 evaluation in a fresh directory. Without that flag it loads the packaged original predictions and the later diagnostics.

The original 2025 evaluation is the historical result. Later cells are labeled post-evaluation.


In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config" / "research.yaml").exists():
    ROOT = Path.cwd().resolve()
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "config" / "research.yaml").exists():
            os.chdir(parent)
            ROOT = parent
            break
print("project", ROOT)

candidates = [
    Path(os.environ["BDB_DATA_ROOT"]) if os.environ.get("BDB_DATA_ROOT") else None,
    ROOT / "nfl-big-data-bowl-2027",
    Path("/kaggle/input/nfl-big-data-bowl-2027"),
]
data = next(p for p in candidates if p is not None and (p / "players.csv").exists())
print("data", data)
print("csv count", len(list(data.glob("*.csv"))))


In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))
import bdb2027
from bdb2027.config import scientific_fingerprint, presentation_fingerprint
print("bdb2027", Path(bdb2027.__file__).resolve())
print("scientific", scientific_fingerprint()[:16])
print("presentation", presentation_fingerprint()[:16])
manifest = json.loads(Path("archive/original_evaluation/freeze_manifest.json").read_text())
print("original frozen_at", manifest["frozen_at"])
print("original code_hash", manifest["code_hash"][:16])
print("train", manifest["n_train_players"], "eval routes recorded separately")


In [ ]:
# Full reproduction. Skipped unless BDB_EXECUTE=1, because it refits from raw tracking.
# Outputs go to BDB2027_WORK so the archived freeze is not overwritten.
import subprocess, sys
if os.environ.get("BDB_EXECUTE") == "1":
    work = os.environ.get("BDB2027_WORK", "/tmp/bdb_notebook_repro")
    env = os.environ.copy()
    env["BDB2027_WORK"] = work
    steps = [
        [sys.executable, "-m", "bdb2027", "extract", "--cohorts", "2023", "2024"],
        [sys.executable, "-m", "bdb2027", "develop"],
        [sys.executable, "-m", "bdb2027", "freeze"],
        [sys.executable, "-m", "bdb2027", "extract", "--cohorts", "2025"],
        [sys.executable, "-m", "bdb2027", "evaluate-heldout", "--cohort", "2025"],
    ]
    for step in steps:
        print("RUN", " ".join(step))
        subprocess.check_call(step, env=env)
    print("fresh work", work)
else:
    print("Display mode. Run with BDB_EXECUTE=1 to refit from the CSVs.")


In [ ]:
pred = pd.read_csv("outputs/predictions.csv")
hold = pred.loc[pred["split"] == "holdout_2025"]
print("players", hold["nfl_id"].nunique(), "routes", len(hold), "dupes", hold.duplicated(["nfl_id","game_id","play_id"]).sum())

def player_mae(frame, col):
    g = frame.groupby("nfl_id").agg(y=("y","mean"), p=(col,"mean"))
    return float((g.y - g.p).abs().mean())

m0, m1, m2 = player_mae(hold,"pred_M0"), player_mae(hold,"pred_M1"), player_mae(hold,"pred_M2")
print("2025 player MAE", round(m0,4), round(m1,4), round(m2,4), "delta", round(m2-m1,4))
print("This is the original evaluated comparison.")


In [ ]:
rel = json.loads(Path("outputs/post_evaluation/reliability.json").read_text())
print(rel["label"])
print("ICC", rel["icc_anova"]["icc"])
print("SD of means", rel["sd_of_receiver_means"], "latent between", rel["latent_between_sd"])
pen = json.loads(Path("outputs/post_evaluation/common_penalty.json").read_text())
print(pen["label"])
print("shared alpha", pen["alpha"], "retrospective delta", pen["delta_m2_minus_m1"])
print(pd.read_csv("outputs/post_evaluation/prediction_decomposition.csv")[["role","total_m2_minus_m1","direct_trait_term","shared_features_after_each_models_scaling","reconstruction_error"]].to_string(index=False))


Figures below are the ones cited in `writeup/draft.md`. Figure 3 is the post-evaluation display of the original three-class errors. It does not refit the 2025 model.


In [ ]:
from IPython.display import Image, display
for name in ["fig1_exit_distance.png", "fig2_support.png", "model_comparison_reviewed.png"]:
    display(Image(filename=f"outputs/figures/{name}"))
